# 01. Comprehensive Data Cleaning & Sensor Bounds Auditing
## Water Quality Prediction via Gradient Boosting & Deep Neural Networks (Phase 1)

### Data Cleaning Workflow:
1. **Dataset Dimensions & Schema:** Calculate total rows, total columns, data types, and memory footprint.
2. **Missing / Empty Value Audit:** Quantify null counts, missing percentages, and left-censored detection limits.
3. **Duplicate Detection & Removal:** Identify and eliminate duplicated observation records.
4. **Physical Sensor Bounds Auditing:** Enforce thermodynamic laws (pH in [0, 14], Temperature in [-5, 50] deg C, DO >= 0).
5. **Non-Destructive Anomaly Protocol:** Replace invalid sensor spikes with NaN and append boolean fault indicators.
6. **Missing Value Imputation:** Thoroughly fill and impute all empty/missing values using grouped and robust medians.
7. **Before vs After Verification:** Validate zero remaining nulls and export clean dataset.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

ROOT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT_DIR) not in sys.path:
    sys.path.append(str(ROOT_DIR))

from src.data.clean import (
    inspect_dataset_shape_and_nulls,
    remove_duplicate_records,
    audit_physical_bounds,
    flag_and_clean_sensor_anomalies,
    fill_missing_values,
    clean_water_quality_dataset,
    DEFAULT_PHYSICAL_BOUNDS
)

plt.style.use("seaborn-v0_8-whitegrid")
%matplotlib inline
print("Data cleaning environment initialized successfully!")

---
## Step 1: Ingest Raw Telemetry & Calculate Dimensions
Calculate total number of rows, number of columns, column schema, and memory usage.

In [ ]:
dataset_path = ROOT_DIR / "data" / "raw" / "Combined_dataset.csv"
df_raw = pd.read_csv(dataset_path, nrows=150000)

# Calculate dimensions
n_rows, n_cols = df_raw.shape
mem_mb = df_raw.memory_usage(deep=True).sum() / (1024 * 1024)

print(f"==================================================")
print(f"  RAW DATASET DIMENSIONS & INVENTORY")
print(f"==================================================")
print(f"  Total Number of Rows    : {n_rows:,}")
print(f"  Total Number of Columns : {n_cols}")
print(f"  Memory Footprint        : {mem_mb:.2f} MB")
print(f"==================================================
")

# Display column types and non-null inventory
dim_summary = pd.DataFrame({
    "Column Name": df_raw.columns,
    "Data Type": [str(t) for t in df_raw.dtypes],
    "Non-Null Count": df_raw.notna().sum().values,
    "Null Count": df_raw.isna().sum().values,
    "Unique Values": df_raw.nunique().values
})
display(dim_summary)

---
## Step 2: Empty / Missing Value & Detection Limit Inspection
In environmental monitoring, missing data appears in two distinct forms:
1. Explicit empty values (, , empty strings).
2. Left-censored values (), where concentrations fall below the instrument detection limit.

In [ ]:
# Calculate explicit missingness
null_counts = df_raw.isna().sum()
null_pct = (df_raw.isna().mean() * 100).round(3)

missing_table = pd.DataFrame({
    "Missing Count": null_counts,
    "Missing Percentage (%)": null_pct
})
print(f"Total Explicit Missing Cells in Raw Load: {null_counts.sum()}")
display(missing_table.T)

# Inspect instrument detection limits (zero readings in chemical species)
nutrient_species = [
    "Ammonia (mg/l)", "Biochemical Oxygen Demand (mg/l)",
    "Orthophosphate (mg/l)", "Nitrogen (mg/l)", "Nitrate (mg/l)"
]

detection_limits = []
for col in nutrient_species:
    zeros = (df_raw[col] == 0.0).sum()
    detection_limits.append({
        "Chemical Species": col,
        "Zero / Censored Count": zeros,
        "Censored Percentage (%)": round((zeros / len(df_raw)) * 100, 2),
        "Minimum Positive Reading": df_raw[df_raw[col] > 0][col].min()
    })
display(pd.DataFrame(detection_limits))

---
## Step 3: Duplicate Observation Detection & Removal
Duplicate telemetry logs can occur during sensor re-transmissions or network retry surges.

In [ ]:
initial_rows = len(df_raw)
num_duplicates = df_raw.duplicated().sum()
print(f"Number of Exact Duplicate Rows Detected: {num_duplicates:,}")

df_dedup, dropped_dups = remove_duplicate_records(df_raw)
print(f"Row count before de-duplication : {initial_rows:,}")
print(f"Duplicate records eliminated    : {dropped_dups:,}")
print(f"Row count after de-duplication  : {len(df_dedup):,}")

---
## Step 4: Physical Sensor Bounds Auditing & Anomaly Flagging
Enforce thermodynamic constraints:
- pH in [0, 14]
- Temperature in [-5, 50] deg C
- Dissolved Oxygen in [0, 30] mg/L

Readings violating physical laws are flagged as sensor faults () and converted to  for controlled imputation.

In [ ]:
bounds_report = audit_physical_bounds(df_dedup)
print("=== Physical Bounds Diagnostic Report ===")
display(bounds_report)

# Execute non-destructive flagging protocol
df_flagged, anomaly_summary = flag_and_clean_sensor_anomalies(df_dedup, action="nan_and_flag")
print("
Anomalies identified and converted to NaN for imputation:")
for col, count in anomaly_summary.items():
    if count > 0:
        print(f"  - {col}: {count} impossible readings converted to NaN")

fault_flag_cols = [c for c in df_flagged.columns if c.endswith("_is_sensor_fault")]
print(f"
Appended {len(fault_flag_cols)} boolean sensor fault indicators:", fault_flag_cols)
print(f"New column count after flagging: {df_flagged.shape[1]} (was {df_dedup.shape[1]})")

---
## Step 5: Filling / Imputing Empty & Missing Values
Now we impute all empty values (s) generated from sensor fault conversions and missing telemetry:
- We use a **grouped median strategy** based on  (River, Lake, Effluent, etc.).
- Global median is applied as a fallback to guarantee zero remaining empty values.

In [ ]:
# 1. Inspect empty values BEFORE filling
nulls_before = df_flagged.isna().sum()
print("=== Empty Value Count BEFORE Imputation ===")
print(nulls_before[nulls_before > 0])
print(f"Total empty cells to be filled: {nulls_before.sum():,}
")

# 2. Impute and fill missing values
df_imputed, filled_counts = fill_missing_values(
    df_flagged,
    numeric_strategy="median",
    group_col="Waterbody Type"
)

# 3. Verify empty values AFTER filling
nulls_after = df_imputed.isna().sum().sum()
print("=== Empty Value Count AFTER Imputation ===")
print(f"Total Remaining Missing Cells: {nulls_after}")
assert nulls_after == 0, "ERROR: Missing values still exist!"
print("SUCCESS: 100% of empty and corrupted values have been filled!")

---
## Step 6: Before-and-After Cleaning Diagnostic Comparison
Comprehensive side-by-side audit of dataset quality before and after cleaning.

In [ ]:
before_after_df = pd.DataFrame({
    "Cleaning Metric": [
        "Total Number of Rows",
        "Total Number of Columns",
        "Duplicate Rows Present",
        "Impossible pH Readings (>14 or <0)",
        "Impossible Temperatures (>50 C)",
        "Negative DO Readings (<0 mg/l)",
        "Empty / Null Values Remaining",
        "Sensor Fault Indicators Created"
    ],
    "Before Cleaning (Raw)": [
        f"{len(df_raw):,}",
        f"{df_raw.shape[1]}",
        f"{num_duplicates:,}",
        f"{(df_raw['pH (ph units)'] > 14).sum():,}",
        f"{(df_raw['Temperature (cel)'] > 50).sum():,}",
        f"{(df_raw['Dissolved Oxygen (mg/l)'] < 0).sum():,}",
        f"{df_raw.isna().sum().sum():,}",
        "0"
    ],
    "After Cleaning": [
        f"{len(df_imputed):,}",
        f"{df_imputed.shape[1]}",
        "0",
        "0 (flagged & imputed)",
        "0 (flagged & imputed)",
        "0 (flagged & imputed)",
        "0 (100% filled)",
        f"{len(fault_flag_cols)}"
    ]
})
display(before_after_df)

---
## Step 7: Export Cleaned Dataset
Persist the clean, validated dataset to  for downstream feature engineering and model training.

In [ ]:
output_processed_path = ROOT_DIR / "data" / "processed" / "cleaned_water_sample.csv"
output_processed_path.parent.mkdir(parents=True, exist_ok=True)

df_imputed.to_csv(output_processed_path, index=False)
print(f"Cleaned dataset successfully exported to: {output_processed_path}")
print(f"Exported Dimensions: {df_imputed.shape[0]:,} rows x {df_imputed.shape[1]} columns.")